In [1]:
# Task 1: Vehicle_ID Trimming & Canonical Case
import pandas as pd

df = pd.read_csv("car_rental_dirty_dataset_new1.csv")


1. Vehicle_ID trimming and canonical case.

In [2]:
unique_vid = df['Vehicle_ID'].nunique()

# Inspect raw Vehicle_ID values
print(f"Shape: {df.shape}")
print(f"Unique Vehicle_IDs (raw): {unique_vid}")
print(sorted(df['Vehicle_ID'].dropna().unique()))
print(f"\n{df['Vehicle_ID'].value_counts().sort_index()}")

Shape: (5000, 21)
Unique Vehicle_IDs (raw): 601
[' CAR-008 ', ' CAR-009 ', ' CAR-019 ', ' CAR-041 ', ' CAR-046 ', ' CAR-056 ', ' CAR-058 ', ' CAR-059 ', ' CAR-062 ', ' CAR-070 ', ' CAR-078 ', ' CAR-079 ', ' CAR-084 ', ' CAR-091 ', ' CAR-099 ', ' CAR-103 ', ' CAR-105 ', ' CAR-108 ', ' CAR-115 ', ' CAR-120 ', ' CAR-124 ', ' CAR-130 ', ' CAR-145 ', ' CAR-182 ', ' CAR-189 ', ' CAR-201 ', ' CAR-216 ', ' CAR-222 ', ' CAR-240 ', ' CAR-255 ', ' CAR-258 ', ' CAR-259 ', ' CAR-264 ', ' CAR-285 ', ' CAR-298 ', ' CAR-348 ', ' CAR-349 ', ' CAR-352 ', ' CAR-358 ', ' CAR-367 ', ' CAR-390 ', ' CAR-401 ', ' CAR-406 ', ' CAR-411 ', ' CAR-425 ', ' CAR-432 ', ' CAR-439 ', ' CAR-443 ', ' CAR-445 ', ' CAR-448 ', ' CAR-460 ', ' CAR-461 ', ' CAR-491 ', ' CAR-511 ', ' CAR-515 ', ' CAR-519 ', ' CAR-535 ', ' CAR-540 ', ' CAR-541 ', ' CAR-545 ', ' CAR-547 ', ' CAR-557 ', ' CAR-562 ', ' CAR-566 ', ' CAR-570 ', ' CAR-573 ', ' CAR-574 ', ' CAR-580 ', ' CAR-585 ', ' CAR-600 ', 'CAR 025', 'CAR 030', 'CAR 032', 'CAR 038

In [3]:
# Clean Vehicle_ID: trim whitespace, normalize separators, canonical uppercase

# Strip leading/trailing whitespace
df['Vehicle_ID'] = df['Vehicle_ID'].str.strip()

# Normalize separators: double hyphens -> single, underscores -> hyphens
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace('--', '-', regex=False)
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace('_', '-', regex=False)

# Insert hyphen between letters and digits where missing (e.g., CAR08 -> CAR-08)
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace(r'([A-Za-z])\s+(\d)', r'\1-\2', regex=True)
df['Vehicle_ID'] = df['Vehicle_ID'].str.replace(r'([A-Za-z])(\d)', r'\1-\2', regex=True)

# Convert to uppercase canonical form
df['Vehicle_ID'] = df['Vehicle_ID'].str.upper()

# Fix CAR-004 -> CAR-04 to match 2-digit format
df['Vehicle_ID'] = df['Vehicle_ID'].replace('CAR-004', 'CAR-04')

# Convert 'UNKNOWN' to NaN, flag invalid IDs
df['Vehicle_ID_Invalid'] = df['Vehicle_ID'].isna() | (df['Vehicle_ID'] == 'UNKNOWN')
df.loc[df['Vehicle_ID'] == 'UNKNOWN', 'Vehicle_ID'] = pd.NA

print("Cleaned Vehicle_IDs:")
print(sorted(df['Vehicle_ID'].dropna().unique()))
print(f"\n{df['Vehicle_ID'].value_counts().sort_index()}")

Cleaned Vehicle_IDs:
['CAR-001', 'CAR-002', 'CAR-003', 'CAR-005', 'CAR-006', 'CAR-007', 'CAR-008', 'CAR-009', 'CAR-010', 'CAR-011', 'CAR-012', 'CAR-013', 'CAR-014', 'CAR-015', 'CAR-016', 'CAR-017', 'CAR-018', 'CAR-019', 'CAR-020', 'CAR-021', 'CAR-022', 'CAR-023', 'CAR-024', 'CAR-025', 'CAR-026', 'CAR-027', 'CAR-028', 'CAR-029', 'CAR-030', 'CAR-031', 'CAR-032', 'CAR-033', 'CAR-034', 'CAR-035', 'CAR-036', 'CAR-037', 'CAR-038', 'CAR-039', 'CAR-04', 'CAR-040', 'CAR-041', 'CAR-042', 'CAR-043', 'CAR-044', 'CAR-045', 'CAR-046', 'CAR-047', 'CAR-048', 'CAR-049', 'CAR-050', 'CAR-051', 'CAR-052', 'CAR-053', 'CAR-054', 'CAR-055', 'CAR-056', 'CAR-057', 'CAR-058', 'CAR-059', 'CAR-060', 'CAR-061', 'CAR-062', 'CAR-063', 'CAR-064', 'CAR-065', 'CAR-066', 'CAR-067', 'CAR-068', 'CAR-069', 'CAR-070', 'CAR-071', 'CAR-072', 'CAR-073', 'CAR-074', 'CAR-075', 'CAR-076', 'CAR-077', 'CAR-078', 'CAR-079', 'CAR-080', 'CAR-081', 'CAR-082', 'CAR-083', 'CAR-084', 'CAR-085', 'CAR-086', 'CAR-087', 'CAR-088', 'CAR-089', 

In [4]:
# Validate: all non-null IDs should match CAR-XX or CAR-XXX
valid_ids = df['Vehicle_ID'].dropna()
pattern_match = valid_ids.str.match(r'^CAR-\d{2,3}$')

print(f"Valid IDs: {len(valid_ids)} | Matching pattern: {pattern_match.sum()} | Non-matching: {(~pattern_match).sum()}")
if (~pattern_match).any():
    print(f"Non-matching: {valid_ids[~pattern_match].unique()}")

print(f"\nUnique Vehicle_IDs — Before: {unique_vid}, After: {df['Vehicle_ID'].nunique()}")
print(f"Invalid (NaN) Vehicle_IDs: {df['Vehicle_ID'].isna().sum()}")

Valid IDs: 4988 | Matching pattern: 4988 | Non-matching: 0

Unique Vehicle_IDs — Before: 601, After: 600
Invalid (NaN) Vehicle_IDs: 12


2. Timestamp normalization; invalid minutes fix; duration must be positive.

In [5]:
import re
def fix_invalid_minutes(ts):
    if pd.isna(ts):
        return ts  
    match = re.search(r'(\d{1,2}):(\d{2})', str(ts))
    
    if match:
        hour = int(match.group(1))
        minute = int(match.group(2))
        
        if minute > 59:
            hour += minute // 60
            minute = minute % 60
        
        if hour > 23:
            hour = hour % 24
            
        ts = re.sub(r'\d{1,2}:\d{2}', f"{hour}:{minute:02d}", str(ts))
            
    return ts

In [6]:
df["Pickup_TS"] = df["Pickup_TS"].apply(fix_invalid_minutes)
df["Return_TS"] = df["Return_TS"].apply(fix_invalid_minutes)
df["Booking_TS"] = df["Booking_TS"].apply(fix_invalid_minutes)
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,...,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid
0,RES-00001,CUST-1407,CAR-151,Luxury,Completed,06/19/2026 0:59,2026-06-27T0:59,2026-06-27T15:59,40729,40787,...,NaN,mumbai,12.948007,77.635873,NaN,netbanking,DL-1170728882,NaN,Navigation system malfunction reported,False
1,RES-00002,CUST-1555,CAR-538,SUV,Cancelled,2026-05-22T11:21,2026-05-27T11:21,2026-05-30T3:21,54220,"54,561",...,WELCOME5,delhi,12.889665,77.635561,134 km/h,card,DL-8720768076,Minor,AC performance slightly low,False
2,RES-00003,CUST-0100,CAR-118,Sedan,No_Show,2026/06/19 5:29,2026/06/22 5:29,25-06-2026 2:29,33021 km,33149km,...,WELCOME5,CHN,12.872257,77.623025,NaN,wallet,DL-2540682838,Major,Customer requested early pickup,False
3,RES-00004,CUST-1480,CAR-085,NaN,Completed,2026-06-04T22:10,2026-06-07T22:10,2026/6/10 19:01,76471,"76,415",...,NaN,MUM,12.856798,77.618630,fast,UPI,DL-3314017702,NaN,Vehicle returned late due to traffic,False
4,RES-00005,CUST-0696,CAR-292,unknown,Cancelled,05/16/2026 9:24,05/23/2026 9:24,23-05-2026 11:24,45940,"46,332",...,DISC20,CHN,12.852438,77.640421,NaN,upi,DL-4018623004,NaN,Vehicle returned late due to traffic,False
5,RES-00006,CUST-0361,CAR-211,Sedan,Completed,2026-03-09T3:03,2026-03-14 3:03,16-03-2026 1:03,"12,565",12894km,...,NaN,Mumbai,12.905509,77.618185,NaN,CARD,DL-5961002183,NaN,AC performance slightly low,False
6,RES-00007,CUST-1834,CAR-164,unknown,Cancelled,2025-12-28 0:00,2026/01/01 0:00,2026/1/2 21:17,19508 km,"19,429",...,NEW10,Bengaluru,12.927118,77.637018,NaN,upi,DL-2941631094,NaN,Car returned in good condition,False
7,RES-00008,CUST-1613,CAR-149,SUV,Cancelled,2026-01-25T0:00,2026/2/1 1:04,02/02/2026 11:00,17715 km,17862km,...,SAVE50,Mumbai,12.911117,77.557031,140,upi,DL-9482900041,Major,Vehicle returned late due to traffic,False
8,RES-00009,CUST-0331,CAR-018,Luxury,No_Show,2026-05-01T5:01,2026-05-09 5:01,05/10/2026 3:01,66473 km,66546km,...,WELCOME5,delhi,12.946923,77.571583,69 km/h,UPI,DL-3510206236,NaN,Customer reported minor scratch on door,False
9,RES-00010,CUST-0880,CAR-223,Sedan,Completed,2026/5/29 19:09,2026-06-08T18:56,2026/06/10 10:56,"16,965",17368km,...,WELCOME5,MUM,12.945888,77.632509,76kmh,CARD,DL-1715906654,Major,Car returned in good condition,False


In [7]:
df["Pickup_TS"] = df["Pickup_TS"].str.replace("/", "-", regex=False).str.replace("T", " ", regex=False)
df["Return_TS"] = df["Return_TS"].str.replace("/", "-", regex=False).str.replace("T", " ", regex=False)
df["Booking_TS"] = df["Booking_TS"].str.replace("/", "-", regex=False).str.replace("T", " ", regex=False)

In [8]:
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"], format='mixed', dayfirst=True)
df["Return_TS"] = pd.to_datetime(df["Return_TS"], format='mixed', dayfirst=True)
df["Booking_TS"] = pd.to_datetime(df["Booking_TS"], format='mixed', dayfirst=True)

In [9]:
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,...,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid
0,RES-00001,CUST-1407,CAR-151,Luxury,Completed,2026-06-19 00:59:00,2026-06-27 00:59:00,2026-06-27 15:59:00,40729,40787,...,NaN,mumbai,12.948007,77.635873,NaN,netbanking,DL-1170728882,NaN,Navigation system malfunction reported,False
1,RES-00002,CUST-1555,CAR-538,SUV,Cancelled,2026-05-22 11:21:00,2026-05-27 11:21:00,2026-05-30 03:21:00,54220,"54,561",...,WELCOME5,delhi,12.889665,77.635561,134 km/h,card,DL-8720768076,Minor,AC performance slightly low,False
2,RES-00003,CUST-0100,CAR-118,Sedan,No_Show,2026-06-19 05:29:00,2026-06-22 05:29:00,2026-06-25 02:29:00,33021 km,33149km,...,WELCOME5,CHN,12.872257,77.623025,NaN,wallet,DL-2540682838,Major,Customer requested early pickup,False
3,RES-00004,CUST-1480,CAR-085,NaN,Completed,2026-04-06 22:10:00,2026-07-06 22:10:00,2026-10-06 19:01:00,76471,"76,415",...,NaN,MUM,12.856798,77.618630,fast,UPI,DL-3314017702,NaN,Vehicle returned late due to traffic,False
4,RES-00005,CUST-0696,CAR-292,unknown,Cancelled,2026-05-16 09:24:00,2026-05-23 09:24:00,2026-05-23 11:24:00,45940,"46,332",...,DISC20,CHN,12.852438,77.640421,NaN,upi,DL-4018623004,NaN,Vehicle returned late due to traffic,False
5,RES-00006,CUST-0361,CAR-211,Sedan,Completed,2026-09-03 03:03:00,2026-03-14 03:03:00,2026-03-16 01:03:00,"12,565",12894km,...,NaN,Mumbai,12.905509,77.618185,NaN,CARD,DL-5961002183,NaN,AC performance slightly low,False
6,RES-00007,CUST-1834,CAR-164,unknown,Cancelled,2025-12-28 00:00:00,2026-01-01 00:00:00,2026-02-01 21:17:00,19508 km,"19,429",...,NEW10,Bengaluru,12.927118,77.637018,NaN,upi,DL-2941631094,NaN,Car returned in good condition,False
7,RES-00008,CUST-1613,CAR-149,SUV,Cancelled,2026-01-25 00:00:00,2026-01-02 01:04:00,2026-02-02 11:00:00,17715 km,17862km,...,SAVE50,Mumbai,12.911117,77.557031,140,upi,DL-9482900041,Major,Vehicle returned late due to traffic,False
8,RES-00009,CUST-0331,CAR-018,Luxury,No_Show,2026-01-05 05:01:00,2026-09-05 05:01:00,2026-10-05 03:01:00,66473 km,66546km,...,WELCOME5,delhi,12.946923,77.571583,69 km/h,UPI,DL-3510206236,NaN,Customer reported minor scratch on door,False
9,RES-00010,CUST-0880,CAR-223,Sedan,Completed,2026-05-29 19:09:00,2026-08-06 18:56:00,2026-10-06 10:56:00,"16,965",17368km,...,WELCOME5,MUM,12.945888,77.632509,76kmh,CARD,DL-1715906654,Major,Car returned in good condition,False


3. Odometer numeric extraction and unit unification (km)

In [10]:
def clean_odometer(value):

    if pd.isna(value):
        return pd.NA

    # convert to string
    value = str(value)

    # remove commas (12,519 → 12519)
    value = value.replace(",", "")

    # remove non-numeric characters (remove km etc.)
    value = re.sub(r'[^0-9.]', '', value)

    if value == "":
        return pd.NA

    return float(value)
# Apply function to both odometer columns
df['Odo_Start'] = df['Odo_Start'].apply(clean_odometer)
df['Odo_End'] = df['Odo_End'].apply(clean_odometer)

print(df[['Odo_Start','Odo_End']].head(10))

   Odo_Start  Odo_End
0    40729.0  40787.0
1    54220.0  54561.0
2    33021.0  33149.0
3    76471.0  76415.0
4    45940.0  46332.0
5    12565.0  12894.0
6    19508.0  19429.0
7    17715.0  17862.0
8    66473.0  66546.0
9    16965.0  17368.0


4. Fuel level normalization (50%→0.5)

In [11]:
# Remove % symbol
df["Fuel_Level"] = df["Fuel_Level"].astype(str).str.replace("%", "")

# Convert column to numeric
df["Fuel_Level"] = pd.to_numeric(df["Fuel_Level"], errors="coerce")

# Convert percentage values to fraction
df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] = df["Fuel_Level"] / 100

df['Fuel_Level'].unique()

array([0.75, 0.5 , 0.1 , 0.3 , 1.  ,  nan, 0.25])

5. Rate parsing to numeric daily rate; currency normalization.

In [12]:
usd_to_inr = 92
eur_to_inr = 106

def clean_rate(val):

    if pd.isna(val):
        return pd.NA

    val = str(val).replace(",", "").strip()

    # USD values
    if "USD" in val or "$" in val:
        num = re.findall(r'\d+', val)
        if num:
            return int(num[0]) * usd_to_inr

    # EURO values
    if "EUR" in val or "€" in val:
        num = re.findall(r'\d+', val)
        if num:
            return int(num[0]) * eur_to_inr

    # INR values
    num = re.findall(r'\d+', val)
    if num:
        return int(num[0])

    return pd.NA


df["Rate"] = df["Rate"].apply(clean_rate)

In [13]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid
0,RES-00001,CUST-1407,CAR-151,Luxury,Completed,2026-06-19 00:59:00,2026-06-27 00:59:00,2026-06-27 15:59:00,40729.0,40787.0,0.75,2300,NaN,mumbai,12.948007,77.635873,NaN,netbanking,DL-1170728882,NaN,Navigation system malfunction reported,False
1,RES-00002,CUST-1555,CAR-538,SUV,Cancelled,2026-05-22 11:21:00,2026-05-27 11:21:00,2026-05-30 03:21:00,54220.0,54561.0,0.50,1840,WELCOME5,delhi,12.889665,77.635561,134 km/h,card,DL-8720768076,Minor,AC performance slightly low,False
2,RES-00003,CUST-0100,CAR-118,Sedan,No_Show,2026-06-19 05:29:00,2026-06-22 05:29:00,2026-06-25 02:29:00,33021.0,33149.0,0.10,2300,WELCOME5,CHN,12.872257,77.623025,NaN,wallet,DL-2540682838,Major,Customer requested early pickup,False
3,RES-00004,CUST-1480,CAR-085,NaN,Completed,2026-04-06 22:10:00,2026-07-06 22:10:00,2026-10-06 19:01:00,76471.0,76415.0,0.10,1800,NaN,MUM,12.856798,77.618630,fast,UPI,DL-3314017702,NaN,Vehicle returned late due to traffic,False
4,RES-00005,CUST-0696,CAR-292,unknown,Cancelled,2026-05-16 09:24:00,2026-05-23 09:24:00,2026-05-23 11:24:00,45940.0,46332.0,0.30,2000,DISC20,CHN,12.852438,77.640421,NaN,upi,DL-4018623004,NaN,Vehicle returned late due to traffic,False
5,RES-00006,CUST-0361,CAR-211,Sedan,Completed,2026-09-03 03:03:00,2026-03-14 03:03:00,2026-03-16 01:03:00,12565.0,12894.0,0.75,2500,NaN,Mumbai,12.905509,77.618185,NaN,CARD,DL-5961002183,NaN,AC performance slightly low,False
6,RES-00007,CUST-1834,CAR-164,unknown,Cancelled,2025-12-28 00:00:00,2026-01-01 00:00:00,2026-02-01 21:17:00,19508.0,19429.0,0.50,1840,NEW10,Bengaluru,12.927118,77.637018,NaN,upi,DL-2941631094,NaN,Car returned in good condition,False
7,RES-00008,CUST-1613,CAR-149,SUV,Cancelled,2026-01-25 00:00:00,2026-01-02 01:04:00,2026-02-02 11:00:00,17715.0,17862.0,0.75,1840,SAVE50,Mumbai,12.911117,77.557031,140,upi,DL-9482900041,Major,Vehicle returned late due to traffic,False
8,RES-00009,CUST-0331,CAR-018,Luxury,No_Show,2026-01-05 05:01:00,2026-09-05 05:01:00,2026-10-05 03:01:00,66473.0,66546.0,0.50,<NA>,WELCOME5,delhi,12.946923,77.571583,69 km/h,UPI,DL-3510206236,NaN,Customer reported minor scratch on door,False
9,RES-00010,CUST-0880,CAR-223,Sedan,Completed,2026-05-29 19:09:00,2026-08-06 18:56:00,2026-10-06 10:56:00,16965.0,17368.0,0.75,1500,WELCOME5,MUM,12.945888,77.632509,76kmh,CARD,DL-1715906654,Major,Car returned in good condition,False


6. City normalization to canonical names.

In [14]:
# Check unique city values before cleaning
print("Cities BEFORE cleaning:")
print(df["City"].unique())

# Step 1: Remove spaces
df["City"] = df["City"].str.strip()

# Step 2: Convert everything to lowercase
df["City"] = df["City"].str.lower()

# Step 3: Replace unknown and missing values with NA
df["City"] = df["City"].replace("unknown", pd.NA)

# Step 4: Map inconsistent names to standard names
city_mapping = {
    "blr": "bengaluru",
    "bangalore": "bengaluru",
    "bengaluru": "bengaluru",

    "mum": "mumbai",
    "bombay": "mumbai",
    "mumbai": "mumbai",

    "del": "delhi",
    "delhi": "delhi",
    "new delhi": "delhi",

    "chn": "chennai",
    "chennai": "chennai",

}

df["City"] = df["City"].replace(city_mapping)

# Step 5: Convert to title case
df["City"] = df["City"].str.title()

df["City"].value_counts()

# Check unique city values after cleaning
print("Cities AFTER cleaning:")
print(df["City"].unique())

Cities BEFORE cleaning:
<StringArray>
[   'mumbai',     'delhi',       'CHN',       'MUM',    'Mumbai', 'Bengaluru',
     'Delhi',   'chennai',       'blr',    'Bombay',   'unknown', 'bangalore',
   'Chennai', 'New Delhi',         nan,       'BLR',     'DELHI']
Length: 17, dtype: str
Cities AFTER cleaning:
<StringArray>
['Mumbai', 'Delhi', 'Chennai', 'Bengaluru', nan]
Length: 5, dtype: str


7. Duplicate reservation dedup (same ID).

In [15]:
print(f"Duplicate Reservation_IDs: {df['Reservation_ID'].duplicated().sum()}")
df = df.drop_duplicates(subset='Reservation_ID', keep='first')
print(f"Shape after dedup: {df.shape}")

Duplicate Reservation_IDs: 473
Shape after dedup: (4527, 22)


8. Return before pickup rule validation.

In [16]:
swap_mask = df["Return_TS"] < df["Pickup_TS"]

df.loc[swap_mask, ["Pickup_TS","Return_TS"]] = \
df.loc[swap_mask, ["Return_TS","Pickup_TS"]].values

In [17]:
df["Duration_Hours"] = (df["Return_TS"] - df["Pickup_TS"]).dt.total_seconds() / 3600

In [18]:
invalid_duration = df[df["Duration_Hours"] < 0]

In [19]:
equal_mask = df["Return_TS"] == df["Pickup_TS"]

In [20]:
df.loc[equal_mask, ["Pickup_TS", "Return_TS"]] = pd.NaT

In [21]:
print(invalid_duration)

Empty DataFrame
Columns: [Reservation_ID, Customer_ID, Vehicle_ID, Vehicle_Class, Booking_Status, Booking_TS, Pickup_TS, Return_TS, Odo_Start, Odo_End, Fuel_Level, Rate, Promo_Code, City, GPS_Lat, GPS_Lon, Speed, Payment, Driver_License, Damage_Flag, Notes, Vehicle_ID_Invalid, Duration_Hours]
Index: []


In [22]:
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid,Duration_Hours
0,RES-00001,CUST-1407,CAR-151,Luxury,Completed,2026-06-19 00:59:00,2026-06-27 00:59:00,2026-06-27 15:59:00,40729.0,40787.0,0.75,2300,NaN,Mumbai,12.948007,77.635873,NaN,netbanking,DL-1170728882,NaN,Navigation system malfunction reported,False,15.000000
1,RES-00002,CUST-1555,CAR-538,SUV,Cancelled,2026-05-22 11:21:00,2026-05-27 11:21:00,2026-05-30 03:21:00,54220.0,54561.0,0.50,1840,WELCOME5,Delhi,12.889665,77.635561,134 km/h,card,DL-8720768076,Minor,AC performance slightly low,False,64.000000
2,RES-00003,CUST-0100,CAR-118,Sedan,No_Show,2026-06-19 05:29:00,2026-06-22 05:29:00,2026-06-25 02:29:00,33021.0,33149.0,0.10,2300,WELCOME5,Chennai,12.872257,77.623025,NaN,wallet,DL-2540682838,Major,Customer requested early pickup,False,69.000000
3,RES-00004,CUST-1480,CAR-085,NaN,Completed,2026-04-06 22:10:00,2026-07-06 22:10:00,2026-10-06 19:01:00,76471.0,76415.0,0.10,1800,NaN,Mumbai,12.856798,77.618630,fast,UPI,DL-3314017702,NaN,Vehicle returned late due to traffic,False,2204.850000
4,RES-00005,CUST-0696,CAR-292,unknown,Cancelled,2026-05-16 09:24:00,2026-05-23 09:24:00,2026-05-23 11:24:00,45940.0,46332.0,0.30,2000,DISC20,Chennai,12.852438,77.640421,NaN,upi,DL-4018623004,NaN,Vehicle returned late due to traffic,False,2.000000
5,RES-00006,CUST-0361,CAR-211,Sedan,Completed,2026-09-03 03:03:00,2026-03-14 03:03:00,2026-03-16 01:03:00,12565.0,12894.0,0.75,2500,NaN,Mumbai,12.905509,77.618185,NaN,CARD,DL-5961002183,NaN,AC performance slightly low,False,46.000000
6,RES-00007,CUST-1834,CAR-164,unknown,Cancelled,2025-12-28 00:00:00,2026-01-01 00:00:00,2026-02-01 21:17:00,19508.0,19429.0,0.50,1840,NEW10,Bengaluru,12.927118,77.637018,NaN,upi,DL-2941631094,NaN,Car returned in good condition,False,765.283333
7,RES-00008,CUST-1613,CAR-149,SUV,Cancelled,2026-01-25 00:00:00,2026-01-02 01:04:00,2026-02-02 11:00:00,17715.0,17862.0,0.75,1840,SAVE50,Mumbai,12.911117,77.557031,140,upi,DL-9482900041,Major,Vehicle returned late due to traffic,False,753.933333
8,RES-00009,CUST-0331,CAR-018,Luxury,No_Show,2026-01-05 05:01:00,2026-09-05 05:01:00,2026-10-05 03:01:00,66473.0,66546.0,0.50,<NA>,WELCOME5,Delhi,12.946923,77.571583,69 km/h,UPI,DL-3510206236,NaN,Customer reported minor scratch on door,False,718.000000
9,RES-00010,CUST-0880,CAR-223,Sedan,Completed,2026-05-29 19:09:00,2026-08-06 18:56:00,2026-10-06 10:56:00,16965.0,17368.0,0.75,1500,WELCOME5,Mumbai,12.945888,77.632509,76kmh,CARD,DL-1715906654,Major,Car returned in good condition,False,1456.000000


9. Payment method standardization (UPI/CARD/CASH/WALLET).

In [23]:
df["Payment"] = df["Payment"].astype(str).str.lower().str.strip()

payment_dict = {
    "upi": "UPI",
    
    "credit card": "CARD",
    "debit card": "CARD",
    "card": "CARD",
    "netbanking": "CARD",
    
    "cash": "CASH",
    
    "wallet": "WALLET",
    
    "-": pd.NA,
    "nan": pd.NA
}

df["Payment"] = df["Payment"].replace(payment_dict)
print(df["Payment"].unique())

<StringArray>
['CARD', 'WALLET', 'UPI', nan, 'CASH']
Length: 5, dtype: str


10. Mileage sanity check (End ≥ Start)

In [24]:
mileage_check = df[['Reservation_ID','Odo_Start','Odo_End']].copy()

# Swap Odo_Start and Odo_End where End < Start (likely data entry error)
odo_swap_mask = df['Odo_End'] < df['Odo_Start']
df.loc[odo_swap_mask, ['Odo_Start', 'Odo_End']] = df.loc[odo_swap_mask, ['Odo_End', 'Odo_Start']].values
print(f"Swapped Odo_Start/Odo_End for {odo_swap_mask.sum()} rows")

# Verify no invalid mileages remain
mileage_check = df[['Reservation_ID','Odo_Start','Odo_End']].copy()
mileage_check['Mileage_Valid'] = mileage_check['Odo_End'] >= mileage_check['Odo_Start']
print(f"Invalid mileages remaining: {(~mileage_check['Mileage_Valid']).sum()}")
mileage_check.head(10)

Swapped Odo_Start/Odo_End for 759 rows
Invalid mileages remaining: 0


,Reservation_ID,Odo_Start,Odo_End,Mileage_Valid
0,RES-00001,40729.0,40787.0,True
1,RES-00002,54220.0,54561.0,True
2,RES-00003,33021.0,33149.0,True
3,RES-00004,76415.0,76471.0,True
4,RES-00005,45940.0,46332.0,True
5,RES-00006,12565.0,12894.0,True
6,RES-00007,19429.0,19508.0,True
7,RES-00008,17715.0,17862.0,True
8,RES-00009,66473.0,66546.0,True
9,RES-00010,16965.0,17368.0,True


11. Refueling event detection vs fuel change and distance.

In [25]:
# Step 1: Normalize Fuel_Level (re-normalize in case of any remaining % values)
df["Fuel_Level"] = df["Fuel_Level"].astype(str).str.replace("%", "", regex=False)
df["Fuel_Level"] = pd.to_numeric(df["Fuel_Level"], errors="coerce")

# Convert percentage values to fraction
df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] = df["Fuel_Level"] / 100


# Step 2: Create temporary dataframe
Refuel_Event = df[["Reservation_ID", "Vehicle_ID", "Odo_Start", "Odo_End", "Fuel_Level"]].copy()


# Step 3: Remove invalid odometer rows (sanity check from UC10)
Refuel_Event = Refuel_Event[Refuel_Event["Odo_End"] >= Refuel_Event["Odo_Start"]]


# Step 4: Calculate distance driven
Refuel_Event["Distance_Driven"] = Refuel_Event["Odo_End"] - Refuel_Event["Odo_Start"]


# Step 5: Detect refuel events (grouped by Vehicle_ID)
Refuel_Event = Refuel_Event.sort_values(["Vehicle_ID", "Odo_Start"])
Refuel_Event["Refuel_Event"] = Refuel_Event.groupby("Vehicle_ID")["Fuel_Level"].diff() > 0

Refuel_Event["Refuel_Event"] = Refuel_Event["Refuel_Event"].map(
    {True: "Refueled", False: "No Refuel"}
)


# Step 6: Replace NaN with "NA" for display
Refuel_Event = Refuel_Event.fillna(pd.NA)


# Step 7: Display
Refuel_Event.head(10)

,Reservation_ID,Vehicle_ID,Odo_Start,Odo_End,Fuel_Level,Distance_Driven,Refuel_Event
556,RES-00557,CAR-001,10329.0,10576.0,1.00,247.0,No Refuel
3130,RES-03131,CAR-001,18654.0,18897.0,0.75,243.0,No Refuel
404,RES-00405,CAR-001,24970.0,25116.0,0.30,146.0,No Refuel
2333,RES-02334,CAR-001,31640.0,31875.0,1.00,235.0,Refueled
1237,RES-01238,CAR-001,62535.0,62932.0,1.00,397.0,No Refuel
2803,RES-02804,CAR-001,70047.0,70145.0,0.50,98.0,No Refuel
1252,RES-01253,CAR-002,14324.0,14390.0,0.75,66.0,No Refuel
2193,RES-02194,CAR-002,14417.0,14567.0,0.30,150.0,No Refuel
58,RES-00059,CAR-002,33502.0,33599.0,0.25,97.0,No Refuel
522,RES-00523,CAR-002,45706.0,45834.0,1.00,128.0,Refueled


12. Vehicle availability overlap checks.

In [26]:
df = df.sort_values(['Vehicle_ID','Pickup_TS'])
df['Prev_Return'] = df.groupby('Vehicle_ID')['Return_TS'].shift(1)
df['Overlap_Flag'] = df['Pickup_TS'] < df['Prev_Return']
overlaps = df[df['Overlap_Flag'] == True]
overlaps
df = df[df['Overlap_Flag'] == False]
df['Overlap_Flag'].sum()

np.int64(0)

13. Damage / Incident Log Linkage to Reservation

In [27]:
#inspect
df[['Reservation_ID','Vehicle_ID','Customer_ID','Damage_Flag','Notes']].head()

#Extract rows where damage was reported
damage_logs = df[
    df['Damage_Flag'].notna() &
    (df['Damage_Flag'] != 'None')
]


#Create a separate incident log dataset
incident_log = damage_logs[[
    'Reservation_ID',
    'Vehicle_ID',
    'Customer_ID',
    'Damage_Flag',
    'Notes'
]].copy()


#Check for records where damage is flagged but notes are missing
missing_notes = incident_log[
    incident_log['Notes'].isna() |
    (incident_log['Notes'].str.strip() == "")
]


print("Damage incidents missing description:", len(missing_notes))


#Detect possible damage mentioned in notes but flag is 'None'
possible_damage = df[
    (df['Damage_Flag'] == 'None') &
    (df['Notes'].str.contains('damage|scratch|dent|crack', case=False, na=False))
]


print("Possible misclassified incidents:", len(possible_damage))


#Display the final incident log dataset
incident_log.head(15)
df['Notes'].unique()


Damage incidents missing description: 352
Possible misclassified incidents: 0


<StringArray>
['Customer reported minor scratch on door',
            'Customer satisfied with ride',
                                       nan,
          'Car returned in good condition',
         'Tyre pressure alert during trip',
  'Navigation system malfunction reported',
         'Customer requested early pickup',
               'Low fuel warning observed',
    'Vehicle returned late due to traffic',
             'AC performance slightly low',
              'Interior cleaning required']
Length: 11, dtype: str

14. Driver license masking and validation (if present).

In [28]:
#validation
def validate_license(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    if re.match(r"DL-\d{10}$", x):
        return x
    else:
        return None

df["Driver_License"] = df["Driver_License"].apply(validate_license)


In [29]:
#masking
def mask_license(x):
    if pd.isna(x):
        return None
    
    return x[:7] + "XXXXXX"

df["Driver_License"] = df["Driver_License"].apply(mask_license)

15. Promo/coupon code validation & expiry checks.

In [30]:
# convert pickup timestamp
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"], errors="coerce")

# promo expiry dictionary
promo_expiry = {
    "NEW10": "2026-03-31",
    "DISC20": "2026-02-28",
    "SAVE50": "2026-06-30",
    "WELCOME5": "2026-01-31"
}

# convert expiry values to datetime
promo_expiry = {k: pd.to_datetime(v) for k, v in promo_expiry.items()}

# map expiry date to dataset
df["Promo_Expiry"] = df["Promo_Code"].map(promo_expiry)

df["Promo_Status"] = "Valid"

# invalid promo codes
df.loc[df["Promo_Code"].notna() & df["Promo_Expiry"].isna(), "Promo_Status"] = "Invalid_Code"

# expired promo codes
df.loc[(df["Promo_Expiry"].notna()) & (df["Pickup_TS"] > df["Promo_Expiry"]), "Promo_Status"] = "Expired"



In [31]:
df.head(30)


,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,Fuel_Level,Rate,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Vehicle_ID_Invalid,Duration_Hours,Prev_Return,Overlap_Flag,Promo_Expiry,Promo_Status
1237,RES-01238,CUST-0446,CAR-001,Hatchback,No_Show,2026-08-01 04:23:00,2026-01-15 04:23:00,2026-01-18 03:23:00,62535.0,62932.0,1.00,1500,NEW10,Bengaluru,12.860193,77.567523,114kmh,NaN,DL-8955XXXXXX,Minor,Customer reported minor scratch on door,False,71.000000,NaT,False,2026-03-31,Valid
3130,RES-03131,CUST-0257,CAR-001,NaN,Cancelled,2026-01-22 12:18:00,2026-01-23 12:18:00,2026-01-25 02:18:00,18654.0,18897.0,0.75,1800,WELCOME5,Bengaluru,12.879495,77.649480,119 km/h,CARD,DL-4156XXXXXX,NaN,Customer satisfied with ride,False,38.000000,2026-01-18 03:23:00,False,2026-01-31,Valid
556,RES-00557,CUST-0029,CAR-001,Hatchback,Completed,2026-02-14 06:33:00,2026-02-18 06:33:00,2026-02-19 08:33:00,10329.0,10576.0,1.00,1500,NEW10,Delhi,12.854050,77.569744,117kmh,CARD,DL-6241XXXXXX,NaN,NaN,False,26.000000,2026-01-25 02:18:00,False,2026-03-31,Valid
404,RES-00405,CUST-0694,CAR-001,Luxury,Completed,2026-04-30 14:16:00,2026-04-05 14:16:00,2026-05-05 22:16:00,24970.0,25116.0,0.30,1800,NEW10,Mumbai,12.913531,77.604304,73kmh,CASH,DL-5092XXXXXX,NaN,Car returned in good condition,False,728.000000,2026-02-19 08:33:00,False,2026-03-31,Expired
2803,RES-02804,CUST-0532,CAR-001,Sedan,Completed,2026-06-18 08:08:00,2026-06-23 08:08:00,2026-06-25 05:14:00,70047.0,70145.0,0.50,2500,WELCOME5,Delhi,12.941481,77.566673,112 km/h,CARD,DL-9286XXXXXX,Major,Tyre pressure alert during trip,False,45.100000,2026-04-21 03:51:00,False,2026-01-31,Expired
522,RES-00523,CUST-0353,CAR-002,unknown,Cancelled,2025-12-30 01:24:00,2026-01-02 01:24:00,2026-02-01 02:00:00,45706.0,45834.0,1.00,<NA>,DISC20,Delhi,12.935368,77.617963,NaN,CARD,DL-2564XXXXXX,Minor,Navigation system malfunction reported,False,720.600000,NaT,False,2026-02-28,Valid
2166,RES-02167,CUST-0079,CAR-002,Sedan,Completed,2026-07-03 11:55:00,2026-03-13 11:55:00,2026-03-15 08:55:00,69759.0,70035.0,0.75,1800,SAVE50,Mumbai,12.932531,77.566649,62 km/h,NaN,DL-3466XXXXXX,Major,Customer requested early pickup,False,45.000000,2026-01-23 06:10:00,False,2026-06-30,Valid
58,RES-00059,CUST-1731,CAR-002,Sedan,No_Show,2026-05-15 13:03:00,2026-05-23 11:09:00,2026-05-23 13:11:00,33502.0,33599.0,0.25,1500,DISC20,Chennai,12.871327,77.629466,fast,CARD,DL-8914XXXXXX,NaN,Low fuel warning observed,False,2.033333,2026-03-15 08:55:00,False,2026-02-28,Expired
2193,RES-02194,CUST-1443,CAR-002,Sedan,Completed,2026-09-06 13:51:00,2026-06-14 13:51:00,2026-06-15 10:51:00,14417.0,14567.0,0.30,1500,SAVE50,Bengaluru,12.866360,77.553843,fast,CARD,DL-5895XXXXXX,Minor,Customer satisfied with ride,False,21.000000,2026-05-23 13:11:00,False,2026-06-30,Valid
2957,RES-02958,CUST-0352,CAR-002,unknown,No_Show,2026-06-27 10:27:00,2026-06-28 10:27:00,2026-06-30 13:27:00,48330.0,48514.0,0.25,2000,NaN,Delhi,12.858027,77.566427,NaN,NaN,DL-1337XXXXXX,Minor,Vehicle returned late due to traffic,False,51.000000,2026-06-15 10:51:00,False,NaT,Valid


16. Telematics GPS join and jitter smoothing.

In [32]:
# 1 check duplicates
df[['GPS_Lat','GPS_Lon']].duplicated().sum()

# 2 handle missing values
df['GPS_Lat'] = df['GPS_Lat'].interpolate()
df['GPS_Lon'] = df['GPS_Lon'].interpolate()

# 3 smooth jitter
df['GPS_Lat'] = df['GPS_Lat'].rolling(window=3, min_periods=1).mean()
df['GPS_Lon'] = df['GPS_Lon'].rolling(window=3, min_periods=1).mean()

17. Speeding/harsh events normalization from telematics.

In [33]:
# Inspect Speed Column
df["Speed"].head()

# Convert Speed Column to String
df["Speed"] = df["Speed"].astype(str)

# Remove Units (kmh and km/h)
df["Speed"] = df["Speed"].str.replace("km/h","", regex=False)
df["Speed"] = df["Speed"].str.replace("kmh","", regex=False)

# Convert "fast" to Numeric Speed

# Since "fast" means driving faster than normal, we convert it to 90 km/h.

df["Speed"] = df["Speed"].replace("fast", 90)

# Convert Entire Column to Numeric
df["Speed"] = pd.to_numeric(df["Speed"], errors="coerce")


# Create Driver Behavior Classification

def classify_speed(speed):

    if pd.isna(speed):
        return "Unknown"

    elif speed <= 80:
        return "Normal Driving"

    elif speed <= 100:
        return "Fast Driving"

    else:
        return "Speeding"


df["Driver_Behavior"] = df["Speed"].apply(classify_speed)


# Verify Result

df[["Speed","Driver_Behavior"]].head(10)

,Speed,Driver_Behavior
1237,114.0,Speeding
3130,119.0,Speeding
556,117.0,Speeding
404,73.0,Normal Driving
2803,112.0,Speeding
522,NaN,Unknown
2166,62.0,Normal Driving
58,90.0,Fast Driving
2193,90.0,Fast Driving
2957,NaN,Unknown


18. PII redaction in notes and feedback.

In [34]:
df['Notes'] = df['Notes'].str.replace(r'\b\d{10}\b', '[REDACTED]', regex=True)
df['Notes'] = df['Notes'].str.replace(r'\S+@\S+', '[REDACTED]', regex=True)
df['Notes'] = df['Notes'].str.replace(r'\b[A-Z0-9]{8,}\b', '[REDACTED]', regex=True)

19. Rate plan mapping to master tariffs.

In [35]:
df['Rate'] = df['Rate'].astype(str)
df['Rate'] = df['Rate'].str.replace('[^0-9.]', '', regex=True)
df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')

20. Tax/GST computation validation.

In [36]:
#USE CASE 20: Tax/GST computation validation.

df["Rate"] = pd.to_numeric(df["Rate"], errors="coerce")
df["Total_Amount"] = (df["Rate"] * 1.18).round(2)

In [37]:
# Save cleaned dataset to CSV
df.to_csv("car_rental_cleaned_dataset.csv", index=False)
print("Cleaned dataset saved to 'car_rental_cleaned_dataset.csv'")
print(f"Final dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Cleaned dataset saved to 'car_rental_cleaned_dataset.csv'
Final dataset shape: (3276, 29)
Columns: ['Reservation_ID', 'Customer_ID', 'Vehicle_ID', 'Vehicle_Class', 'Booking_Status', 'Booking_TS', 'Pickup_TS', 'Return_TS', 'Odo_Start', 'Odo_End', 'Fuel_Level', 'Rate', 'Promo_Code', 'City', 'GPS_Lat', 'GPS_Lon', 'Speed', 'Payment', 'Driver_License', 'Damage_Flag', 'Notes', 'Vehicle_ID_Invalid', 'Duration_Hours', 'Prev_Return', 'Overlap_Flag', 'Promo_Expiry', 'Promo_Status', 'Driver_Behavior', 'Total_Amount']
